# AquaLimit 실제 지역 검증 (Google Colab)

**California Aqueduct MP 195~215 (Lost Hills 주변 31km)** 실제 자료로 지하수·침하 물리모델(MODFLOW 6 + CSUB)을 보정하고, 보정에 쓰지 않은 **GPS 관측으로 예측을 검증**합니다.

| 방식 | 보정에 쓰는 자료 |
|---|---|
| E2 | 관측정 수위만 |
| E3 | 관측정 수위 + 위성(InSAR) |
| E4 | E3 + 불확실성(앙상블 30개) |

- 보정 기간: 2015.7 ~ 2022.12 / 검증 기간: 2023.1 ~ 2026.4
- 전체 실행 시간: Colab 기본(CPU 2개)에서 약 10~15분
- 메뉴 **런타임 → 모두 실행**을 누르면 처음부터 끝까지 돌아갑니다.

## 1. 설치

- `flopy`: Python으로 MODFLOW 모델을 만들고 실행하는 라이브러리
- `mf6`: USGS MODFLOW 6 실행파일 (`get-modflow`가 내려받음)
- `fonts-nanum`: 그래프에 한글을 쓰기 위한 글꼴

In [ ]:
!pip -q install flopy shapely
!get-modflow /usr/local/bin --subset mf6 --quiet
!apt-get -qq install -y fonts-nanum > /dev/null
!mf6 --version

## 2. 코드와 자료 가져오기

GitHub 저장소에서 가져옵니다. (`real/` 폴더의 `real_model.py`, `run_real.py`, `data/`를 커밋·푸시한 뒤 실행하세요.)

푸시하지 않았다면: 왼쪽 파일 패널에 `real` 폴더를 통째로 업로드하고 아래 `git clone` 줄 대신 `%cd /content/real`만 실행해도 됩니다.

In [ ]:
import os
if not os.path.exists("/content/AquaLimit"):
    !git clone -q https://github.com/seoyoooooni/AquaLimit.git /content/AquaLimit
%cd /content/AquaLimit/real
!ls data

## 3. 자료 불러오기

`real_model.load_site()`가 하는 일:
1. **InSAR**: 500m 격자 월별 누적 침하(DWR TRE Altamira) → 모델 격자(1km) 평균. 유전 격자는 제외
2. **관측정**: DWR 정기 측정 수위 → 관측정별 첫 측정 대비 변화(m)
3. **GPS**: 수로를 따라 있는 GPS 5곳(P544, Q198, Q204, Q208, Q213)의 수직 위치 — 검증 전용
4. **강수**: NOAA Lost Hills·Buttonwillow 관측소 → 월 강수 (함양 계산용)
5. **농경지**: 수로 동쪽 격자 = 취수 지역

In [ ]:
import os, time, json, warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import real_model as r
import run_real as rr          # 그림·표 함수, 한글 글꼴 설정

site = r.load_site()
print("모델 격자:", site["ny"], "×", site["nx"], "(1km)")
print("농경지(취수) 격자:", int(site["ag"].sum()))
print("기간:", site["months"][0], "~", site["months"][-1], f"({len(site['months'])}개월)")
for p in ["cal", "val"]:
    i = r.obs_index(site, p, True)
    print(f"{'보정' if p=='cal' else '검증'} 자료: InSAR {i['n_ins']:,}개, 관측정 수위 {i['n_hh']}개, GPS {len(i['gp'])}개")

## 4. 모델 한 번 돌려보기 (사전값)

3개 층: 상부 대수층(0~-60m) / 점토 차수층(-60~-80m) / 하부 취수 대수층(-80~-300m, 안에 압축층 60m).
압축층은 **과거 최저 수위보다 내려가면 비탄성(영구) 압축**, 그 위에서는 탄성 압축(1/10)입니다.

In [ ]:
t = time.time()
S, H1, H3 = r.run_model(site, r.PRIOR_MEAN)
print(f"MODFLOW 1회 실행: {time.time()-t:.1f}초")
plt.figure(figsize=(5,4)); plt.imshow(S[-1], origin="lower", cmap="RdBu_r", vmin=-60, vmax=200)
plt.colorbar(label="누적 침하 (mm)"); plt.title("사전값 모델의 2026.4 누적 침하 (보정 전)"); plt.show()

## 5. 역산 (땅속 성질 추정)

**가우스-뉴턴(레벤버그-마쿼트) 최소제곱**: 관측과 모델의 차이가 가장 작아지도록 파라미터 14개를 고칩니다.
- 구역별 압축성 Sk (3×3 = 9개), 하부층 투수계수, 차수층 연직 투수계수
- 농경지 취수량, 가뭄 해 취수 증가 계수, 함양 비율 (취수 기록이 공개돼 있지 않아 함께 추정)

반복마다 파라미터를 하나씩 조금 바꿔 MODFLOW를 14번 돌려 기울기를 계산합니다(병렬).

In [ ]:
QUICK = os.environ.get("AQ_QUICK") == "1"     # 테스트용(반복 수 줄이기)
N_ITER, N_POST = (1, 4) if QUICK else (6, 30)

print("[E2] 관측정 수위만으로 역산")
th2, cov2, h2 = r.calibrate(site, use_insar=False, n_iter=N_ITER)
print("\n[E3] 관측정 수위 + 위성(InSAR)으로 역산")
th3, cov3, h3 = r.calibrate(site, use_insar=True, n_iter=N_ITER)

## 6. 불확실성 (E4)

E3 추정값 주변의 불확실성(선형화 사후분포)에서 30개를 뽑아 모두 MODFLOW로 돌립니다. 이 30개의 퍼짐이 예측 범위입니다.

In [ ]:
M4, o4 = r.posterior_ensemble(site, th3, cov3, n=N_POST)
M2, o2e = r.posterior_ensemble(site, th2, cov2, n=N_POST, seed=1)
o2, o3 = r.run_model(site, th2), r.run_model(site, th3)
print("앙상블 실행 완료:", len(o4), "개")

## 7. 검증 — 보정에 쓰지 않은 GPS·관측정으로 채점

2023년 이후 DWR InSAR는 GPS와 크게 어긋나서(지반 상승으로 나옴) 검증에 쓰지 않고 **GPS를 정답**으로 씁니다.

In [ ]:
res = {"E2": r.evaluate(site, [o2]), "E3": r.evaluate(site, [o3]), "E4": r.evaluate(site, o4), "E2_ens": r.evaluate(site, o2e),
       "params": {"names": r.NAMES, "prior": r.PRIOR_MEAN.tolist(), "E2": th2.tolist(), "E3": th3.tolist(),
                  "E2_sd": np.sqrt(np.diag(cov2)).tolist(), "E3_sd": np.sqrt(np.diag(cov3)).tolist()},
       "history": {"E2": h2, "E3": h3}, "minutes": None}
tbl = pd.DataFrame({k: {"GPS 침하 오차(mm)": v["val"].get("gps_rmse"), "수위 오차(m)": v["val"].get("head_rmse"),
                        "GPS 90% 범위 포함률": v["val"].get("gps_cov90")} for k, v in res.items() if k.startswith("E")}).T
tbl.round(2)

## 8. 그림과 결과 파일 저장 (`real/results/`)

In [ ]:
rr.OUT.mkdir(exist_ok=True)
json.dump(res, open(rr.OUT / "results.json", "w"), ensure_ascii=False, indent=1, default=float)
rr.figures(site, o2, o3, o4, o2e)
rr.report(res)
from IPython.display import Image, Markdown, display
display(Image(str(rr.OUT / "real-3-gps-validation.png")))
display(Image(str(rr.OUT / "real-4-map-fit.png")))
display(Markdown((rr.OUT / "RESULTS.md").read_text(encoding="utf-8")))

## 9. 결과 내려받기

`real/results/` 폴더를 압축해서 내려받습니다.

In [ ]:
!cd /content/AquaLimit/real && zip -qr results.zip results
try:
    from google.colab import files; files.download("/content/AquaLimit/real/results.zip")
except ImportError:
    print("Colab이 아니면 real/results.zip을 직접 가져가세요.")